# Figures

Generates every figure and equation render cited in the thesis. All outputs are
written as `.jpg` files to the `figures/` subdirectory next to this notebook.

**Inputs**

  - `../sp500_it_processed_full.csv.gz` — long-form panel of 14 features × 47 stocks (only
    `TICKER`, `date`, `PRC`, `RSI`, `log_vol` columns are used here, and only the
    `date` column for the equity-curve x-axis).
  - `results/pipeline_a_test_daily_returns.csv` — daily returns for all five
    strategies on the Pipeline A test window (2022–2024), used by Figure 9.
  - `results/pipeline_b_test_results.csv` — per-window Sharpe for all five
    strategies across the six Pipeline B walk-forward windows, used by Figure 10.

**Outputs (written to `figures/`)**

  - `figure1_pipeline.jpg`               — overall methodological pipeline
  - `figure2_zscore_prc.jpg`             — AAPL closing price, raw vs z-score
  - `figure3_zscore_rsi.jpg`             — AAPL RSI, raw vs z-score
  - `figure4_zscore_logvol.jpg`          — AAPL log volume, raw vs z-score
  - `figure5_ppo_architecture.jpg`       — PPO actor network architecture
  - `figure6_dqn_architecture.jpg`       — DQN Q-network architecture
  - `figure7_pipeline_a.jpg`             — Pipeline A fixed train/val/test split
  - `figure8_pipeline_b.jpg`             — Pipeline B six walk-forward windows
  - `figure9_test_a_equity.jpg`          — Pipeline A test-set equity curves
  - `figure10_test_b_sharpe_bar.jpg`     — Pipeline B per-window Sharpe ratios
  - `eq_zscore.jpg`                      — Equation (1), z-score standardisation
  - `eq_reward.jpg`                      — Equation (2), clipped reward signal
  - `eq_sharpe.jpg`                      — Equation (3), annualised Sharpe ratio
  - `eq_hypotheses.jpg`                  — Equation (4), paired-test hypotheses

The notebook contains no global side effects beyond writing those files; each
cell can be re-run in isolation.


## Setup


In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.dates as mdates
from matplotlib.patches import FancyBboxPatch, Ellipse, Rectangle
from matplotlib.ticker import FuncFormatter

matplotlib.rcParams['font.family'] = 'serif'

# Figures go to ./figures/ next to this notebook
FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

# Data paths (relative to notebook location)
PRICES_GZ      = Path("..") / "sp500_it_processed_full.csv.gz"
RESULTS_DIR    = Path("results")
PIPE_A_RET_CSV = RESULTS_DIR / "pipeline_a_test_daily_returns.csv"
PIPE_B_RES_CSV = RESULTS_DIR / "pipeline_b_test_results.csv"

print(f"Saving figures to: {FIG_DIR.resolve()}")


## Figure 1 — Methodological pipeline


In [ ]:
# ---------- palette ----------
DATA_COLOR  = "#F5B82E"
PREP_COLOR  = "#C5E0B4"
MODEL_COLOR = "#9DC3E6"
EVAL_COLOR  = "#F4B084"
WHITE       = "#FFFFFF"

fig, ax = plt.subplots(figsize=(16, 10), dpi=200)
ax.set_xlim(0, 100); ax.set_ylim(0, 100); ax.axis("off")

def box(x, y, w, h, color, text, fontsize=8, bold=False, ec="black", lw=0.8):
    p = FancyBboxPatch((x, y), w, h,
                       boxstyle="round,pad=0.02,rounding_size=0.4",
                       facecolor=color, edgecolor=ec, linewidth=lw)
    ax.add_patch(p)
    ax.text(x + w/2, y + h/2, text, ha="center", va="center",
            fontsize=fontsize, fontweight="bold" if bold else "normal")

def cylinder(x, y, w, h, color, text, fontsize=8):
    eh = h * 0.20
    ax.add_patch(Rectangle((x, y + eh/2), w, h - eh,
                           facecolor=color, edgecolor="black", linewidth=0.8))
    ax.add_patch(Ellipse((x + w/2, y + eh/2), w, eh,
                         facecolor=color, edgecolor="black", linewidth=0.8))
    ax.add_patch(Ellipse((x + w/2, y + h - eh/2), w, eh,
                         facecolor=color, edgecolor="black", linewidth=0.8))
    ax.text(x + w/2, y + h/2, text, ha="center", va="center", fontsize=fontsize)

def arrow(x1, y1, x2, y2, lw=1.0):
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle="-|>", color="black", lw=lw,
                                mutation_scale=12))

# TIER 1 - DATA + DATA PREP
cylinder(2,  78, 14, 14, DATA_COLOR,
         "S&P 500 IT\n47 constituents\nCRSP daily\n1999-2024", fontsize=8)
cylinder(18, 78, 14, 14, DATA_COLOR,
         "CBOE VIX\n(FRED)\nDaily macro\nvolatility", fontsize=8)
box(34, 78, 22, 14, PREP_COLOR,
    "Feature engineering\n14-feature panel / stock\n"
    "- 5 raw CRSP fields\n- 1 macro (VIX)\n"
    "- 8 technicals: RSI, MACD,\n  BB_PCT, ATR, sigma_60,\n  SMA_20, SMA_50, log_vol",
    fontsize=7.5)
box(58, 78, 18, 14, PREP_COLOR,
    "Train / Val / Test split\n+ z-score standardisation\n"
    "(fit on train, applied\nto val and test)",
    fontsize=8)
box(78, 78, 18, 14, PREP_COLOR,
    "State construction\nDim 705 =\n47 x 14 features\n+ 47 weights\nT = 252,  gamma = 0.99",
    fontsize=8)
arrow(16, 85, 18, 85); arrow(32, 85, 34, 85)
arrow(56, 85, 58, 85); arrow(76, 85, 78, 85)

# TIER 2 - PIPELINES
box(34, 60, 62, 14, WHITE,
    "Pipelines\n\n"
    "Pipeline A - fixed split:  Train 1999-2018  .  Val 2019-2021  .  Test 2022-2024  (27-config grid)\n"
    "Pipeline B - walk-forward:  six rolling annual windows, test 2018-2023  "
    "(hyperparameters retuned per window, 90 trials)",
    fontsize=8)
arrow(67, 78, 67, 74)

# TIER 3 - MODELING HEADER
box(2, 50, 94, 6, MODEL_COLOR,
    "Modeling   (Stable-Baselines3)", fontsize=11, bold=True)
arrow(87, 78, 87, 56); arrow(65, 60, 65, 56)

# TIER 4 - MODELING BODY
box(2,  30, 22, 16, MODEL_COLOR,
    "Reward signal\n\n"
    "r_t = clip(100 . r^p_{t+1}, -10, 10)\n"
    "r^p = w_t . R_{t+1} - c . ||dw||_1\n\n"
    "c = 0.001  (transaction\ncost penalty)",
    fontsize=7.5)
box(26, 30, 22, 16, MODEL_COLOR,
    "PPO\n\nContinuous action space\n47-dim softmax policy\n"
    "Long-only,\nfully invested",
    fontsize=8)
box(50, 30, 22, 16, MODEL_COLOR,
    "DQN\n\nDiscrete action space\n9 fixed portfolio templates\n"
    "(EW + 5 partitions\n+ 3 tilts)",
    fontsize=8)
box(74, 30, 22, 16, MODEL_COLOR,
    "Hyperparameter tuning\n\nSearch over:\n"
    "- training-step budget\n- learning rate\n- random seed",
    fontsize=8)
arrow(13, 50, 13, 46); arrow(37, 50, 37, 46)
arrow(61, 50, 61, 46); arrow(85, 50, 85, 46)

# TIER 5 - EVALUATION
box(2, 2, 94, 24, EVAL_COLOR,
    "Evaluation\n\n"
    "Baselines:  Equal-Weight (1/47, daily rebalance)   .   "
    "Buy-and-Hold (uniform start, drift)   .   Market-Cap Weight (cap-weighted index)\n\n"
    "Metrics:  Annualised Sharpe ratio   .   Annualised return   .   "
    "Annualised volatility   .   Maximum drawdown   .   Turnover\n\n"
    "Statistical test:  Paired one-sided t-test on per-window Sharpe   "
    "(Pipeline B, alpha = 0.05, Equal-Weight as primary benchmark)",
    fontsize=9)
arrow(37, 30, 37, 26); arrow(61, 30, 61, 26)

# LEGEND
legend_patches = [
    mpatches.Patch(facecolor=DATA_COLOR,  edgecolor="black", label="Data"),
    mpatches.Patch(facecolor=PREP_COLOR,  edgecolor="black", label="Data preparation / state"),
    mpatches.Patch(facecolor=MODEL_COLOR, edgecolor="black", label="Modeling"),
    mpatches.Patch(facecolor=EVAL_COLOR,  edgecolor="black", label="Evaluation"),
]
ax.legend(handles=legend_patches, loc="lower right",
          bbox_to_anchor=(1.0, 1.005), fontsize=10, frameon=True,
          ncol=4, handlelength=1.6, handleheight=1.2, columnspacing=1.8)

fig.text(0.5, 0.005,
         "Figure 1: Methodological pipeline. CRSP and VIX data are combined into a 14-feature panel per stock, "
         "standardised and assembled into a 705-D state. PPO and DQN are trained under two evaluation pipelines "
         "(fixed split and walk-forward) and benchmarked against three passive baselines using Sharpe, drawdown, "
         "and a paired t-test.",
         ha="center", fontsize=9, family="serif", wrap=True)

out = FIG_DIR / "figure1_pipeline.jpg"
plt.savefig(out, bbox_inches="tight", dpi=300, facecolor="white")
plt.show()
print(f"Saved {out}")


## Figures 2–4 — Z-score standardisation examples (AAPL)

Three illustrative side-by-side plots showing how train-window z-score
standardisation transforms a representative price series, a bounded momentum
oscillator (RSI), and a heavy-tailed volume series. All scaler parameters
(μ, σ) are estimated on the training window only (≤ 2018-12-31).


In [ ]:
# ── Load AAPL ─────────────────────────────────────────────────────────────
df = pd.read_csv(PRICES_GZ, usecols=['TICKER', 'date', 'PRC', 'RSI', 'log_vol'])
df = df[df['TICKER'] == 'AAPL'].copy()
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

train_end = pd.Timestamp('2018-12-31')
val_end   = pd.Timestamp('2021-12-31')

def zscore(series, train_mask):
    mu    = series[train_mask].mean()
    sigma = series[train_mask].std()
    return (series - mu) / sigma

train_mask = df['date'] <= train_end
df['PRC_z']     = zscore(df['PRC'],     train_mask)
df['RSI_z']     = zscore(df['RSI'],     train_mask)
df['log_vol_z'] = zscore(df['log_vol'], train_mask)

GREY   = '#444444'
BLUE   = '#2166AC'
V_KWARGS = dict(linestyle='--', linewidth=0.9, alpha=0.75)

def fmt_ax(ax, ylabel=''):
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_major_locator(mdates.YearLocator(5))
    ax.tick_params(labelsize=8)
    ax.set_ylabel(ylabel, fontsize=8, family='serif')
    for spine in ax.spines.values():
        spine.set_linewidth(0.6)
    ax.grid(True, linewidth=0.3, alpha=0.5)

def add_boundaries(ax):
    t = ax.get_xaxis_transform()
    ax.axvline(train_end, color='#E31A1C', **V_KWARGS)
    ax.axvline(val_end,   color='#FF7F00', **V_KWARGS)
    ax.text(train_end, 0.97, ' train/val', transform=t, fontsize=7,
            family='serif', color='#E31A1C', ha='left', va='top')
    ax.text(val_end,   0.97, ' val/test',  transform=t, fontsize=7,
            family='serif', color='#FF7F00', ha='left', va='top')

# Figure 2 — Closing Price
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), dpi=300)
fig.suptitle('Figure 2: Closing price (AAPL) — raw and z-score standardised',
             fontsize=10, family='serif')
fig.subplots_adjust(top=0.88, wspace=0.3)
axes[0].plot(df['date'], df['PRC'], color=GREY, linewidth=0.7)
axes[0].set_title('Raw PRC (AAPL)', fontsize=9, family='serif')
fmt_ax(axes[0], 'USD'); add_boundaries(axes[0])
axes[1].plot(df['date'], df['PRC_z'], color=GREY, linewidth=0.7)
axes[1].axhline(0, color='black', linewidth=0.4, linestyle=':')
axes[1].set_title('Z-score standardised PRC (AAPL)', fontsize=9, family='serif')
fmt_ax(axes[1], 'Standard deviations'); add_boundaries(axes[1])
out = FIG_DIR / 'figure2_zscore_prc.jpg'
plt.savefig(out, bbox_inches='tight', dpi=300, facecolor='white')
plt.show(); print(f'Saved {out}')

# Figure 3 — RSI
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), dpi=300)
fig.suptitle('Figure 3: RSI (AAPL) — raw and z-score standardised',
             fontsize=10, family='serif')
fig.subplots_adjust(top=0.88, wspace=0.3)
axes[0].plot(df['date'], df['RSI'], color=GREY, linewidth=0.7)
axes[0].axhline(70, color=BLUE, linewidth=0.5, linestyle='--', alpha=0.6, label='Overbought (70)')
axes[0].axhline(30, color=BLUE, linewidth=0.5, linestyle='--', alpha=0.6, label='Oversold (30)')
axes[0].set_title('Raw RSI (AAPL)', fontsize=9, family='serif')
axes[0].legend(fontsize=7, loc='lower left', framealpha=0.5)
fmt_ax(axes[0], 'RSI (0–100)'); add_boundaries(axes[0])
axes[1].plot(df['date'], df['RSI_z'], color=GREY, linewidth=0.7)
axes[1].axhline(0, color='black', linewidth=0.4, linestyle=':')
axes[1].set_title('Z-score standardised RSI (AAPL)', fontsize=9, family='serif')
fmt_ax(axes[1], 'Standard deviations'); add_boundaries(axes[1])
out = FIG_DIR / 'figure3_zscore_rsi.jpg'
plt.savefig(out, bbox_inches='tight', dpi=300, facecolor='white')
plt.show(); print(f'Saved {out}')

# Figure 4 — Log Volume
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), dpi=300)
fig.suptitle('Figure 4: Log volume (AAPL) — raw and z-score standardised',
             fontsize=10, family='serif')
fig.subplots_adjust(top=0.88, wspace=0.3)
axes[0].plot(df['date'], df['log_vol'], color=GREY, linewidth=0.7)
axes[0].set_title('Raw log(1 + VOL) (AAPL)', fontsize=9, family='serif')
fmt_ax(axes[0], 'log(1 + VOL)'); add_boundaries(axes[0])
axes[1].plot(df['date'], df['log_vol_z'], color=GREY, linewidth=0.7)
axes[1].axhline(0, color='black', linewidth=0.4, linestyle=':')
axes[1].set_title('Z-score standardised log(1 + VOL) (AAPL)', fontsize=9, family='serif')
fmt_ax(axes[1], 'Standard deviations'); add_boundaries(axes[1])
out = FIG_DIR / 'figure4_zscore_logvol.jpg'
plt.savefig(out, bbox_inches='tight', dpi=300, facecolor='white')
plt.show(); print(f'Saved {out}')


## Figure 5 — PPO actor network architecture


In [ ]:
C_INPUT  = '#4472C4'; C_HIDDEN = '#70AD47'; C_OUTPUT = '#ED7D31'
C_LINE   = '#AAAAAA'; C_ARROW  = '#555555'; BG = 'white'

fig, ax = plt.subplots(figsize=(11, 5.5), dpi=300)
ax.set_xlim(0, 11); ax.set_ylim(0, 5.5); ax.axis('off')
fig.patch.set_facecolor(BG)

R, LW = 0.22, 0.6

def draw_nodes(cx, ys, colour, radius=R):
    for y in ys:
        ax.add_patch(plt.Circle((cx, y), radius, facecolor=colour,
                                edgecolor='white', linewidth=1.0, zorder=3))

def connect(x0, ys0, x1, ys1, colour=C_LINE, lw=LW, alpha=0.35):
    for y0 in ys0:
        for y1 in ys1:
            ax.plot([x0 + R, x1 - R], [y0, y1],
                    color=colour, linewidth=lw, alpha=alpha, zorder=1)

def col_label(cx, top_y, lines, fontsize=7.5):
    ax.text(cx, top_y + 0.42, '\n'.join(lines), ha='center', va='bottom',
            fontsize=fontsize, family='serif', fontweight='bold', linespacing=1.3)

def col_sublabel(cx, bot_y, text, fontsize=6.5):
    ax.text(cx, bot_y - 0.42, text, ha='center', va='top',
            fontsize=fontsize, family='serif', color='#666666', style='italic')

X_IN, X_H1, X_H2, X_SOFT, X_OUT = 1.1, 3.2, 5.3, 7.4, 9.6

# Input
N_IN_SHOWN = 5; y_in_top = 4.5; y_in_step = 0.7
ys_in = [y_in_top - i * y_in_step for i in range(N_IN_SHOWN)]
draw_nodes(X_IN, ys_in, C_INPUT)
dot_y = (ys_in[2] + ys_in[3]) / 2
for dy in [-0.18, 0.0, 0.18]:
    ax.plot(X_IN, dot_y + dy, 'o', color='#AAAAAA', markersize=1.8, zorder=4)
col_label(X_IN, y_in_top, ['State', 'Input']); col_sublabel(X_IN, ys_in[-1], '705-D')

# Hidden 1
N_H = 6; y_h_top = 4.55; y_h_step = 0.62
ys_h1 = [y_h_top - i * y_h_step for i in range(N_H)]
connect(X_IN, ys_in, X_H1, ys_h1); draw_nodes(X_H1, ys_h1, C_HIDDEN)
col_label(X_H1, y_h_top, ['Hidden', 'Layer 1']); col_sublabel(X_H1, ys_h1[-1], '64 units, ReLU')

# Hidden 2
ys_h2 = [y_h_top - i * y_h_step for i in range(N_H)]
connect(X_H1, ys_h1, X_H2, ys_h2); draw_nodes(X_H2, ys_h2, C_HIDDEN)
col_label(X_H2, y_h_top, ['Hidden', 'Layer 2']); col_sublabel(X_H2, ys_h2[-1], '64 units, ReLU')

# Softmax
N_SOFT_SHOWN = 5; y_soft_top = 4.45; y_soft_step = 0.68
ys_soft = [y_soft_top - i * y_soft_step for i in range(N_SOFT_SHOWN)]
connect(X_H2, ys_h2, X_SOFT, ys_soft); draw_nodes(X_SOFT, ys_soft, C_OUTPUT)
dot_y = (ys_soft[2] + ys_soft[3]) / 2
for dy in [-0.18, 0.0, 0.18]:
    ax.plot(X_SOFT, dot_y + dy, 'o', color='#AAAAAA', markersize=1.8, zorder=4)
col_label(X_SOFT, y_soft_top, ['Softmax', 'Layer']); col_sublabel(X_SOFT, ys_soft[-1], '47 logits')

# Output
N_OUT_SHOWN = 5; y_out_top = 4.45; y_out_step = 0.68
ys_out = [y_out_top - i * y_out_step for i in range(N_OUT_SHOWN)]
mid_y = (ys_soft[0] + ys_soft[-1]) / 2
ax.annotate('', xy=(X_OUT - R - 0.05, mid_y), xytext=(X_SOFT + R + 0.05, mid_y),
            arrowprops=dict(arrowstyle='->', color=C_ARROW, lw=1.0))
ax.text((X_SOFT + X_OUT) / 2, mid_y + 0.22, 'softmax',
        ha='center', va='bottom', fontsize=6.5, family='serif', color=C_ARROW, style='italic')
draw_nodes(X_OUT, ys_out, C_OUTPUT)
dot_y_o = (ys_out[2] + ys_out[3]) / 2
for dy in [-0.18, 0.0, 0.18]:
    ax.plot(X_OUT, dot_y_o + dy, 'o', color='#AAAAAA', markersize=1.8, zorder=4)
col_label(X_OUT, y_out_top, ['Portfolio', 'Weights'])
col_sublabel(X_OUT, ys_out[-1], r'$w_t \in \Delta^{46}$')

asset_labels = ['Stock 1', 'Stock 2', 'Stock 3', '...', 'Stock 47']
for y, lbl in zip(ys_out, asset_labels):
    ax.text(X_OUT + R + 0.12, y, lbl, ha='left', va='center',
            fontsize=6.5, family='serif', color='#333333')

feat_labels = ['Feature 1', 'Feature 2', 'Feature 3', '...', 'Feature 705']
for y, lbl in zip(ys_in, feat_labels):
    ax.text(X_IN - R - 0.12, y, lbl, ha='right', va='center',
            fontsize=6.5, family='serif', color='#333333')

ax.text(5.5, 0.55,
        '† PPO additionally trains a separate Critic network (same architecture) to estimate V(s).',
        ha='center', va='center', fontsize=6.5, family='serif',
        color='#555555', style='italic')

ax.text(5.5, 5.28, 'Figure 5: PPO Actor network architecture',
        ha='center', va='bottom', fontsize=9, family='serif', style='italic')

plt.tight_layout(pad=0.3)
out = FIG_DIR / 'figure5_ppo_architecture.jpg'
plt.savefig(out, bbox_inches='tight', dpi=300, facecolor=BG)
plt.show(); print(f'Saved {out}')


## Figure 6 — DQN Q-network architecture


In [ ]:
C_INPUT  = '#4472C4'; C_HIDDEN = '#70AD47'; C_OUTPUT = '#ED7D31'
C_SELECT = '#FFC000'; C_LINE   = '#AAAAAA'; C_ARROW  = '#555555'; BG = 'white'
R, LW = 0.22, 0.6

fig, ax = plt.subplots(figsize=(11, 5.5), dpi=300)
ax.set_xlim(0, 11); ax.set_ylim(0, 5.5); ax.axis('off')
fig.patch.set_facecolor(BG)

def draw_nodes(cx, ys, colour, radius=R):
    for y in ys:
        ax.add_patch(plt.Circle((cx, y), radius, facecolor=colour,
                                edgecolor='white', linewidth=1.0, zorder=3))

def connect(x0, ys0, x1, ys1, colour=C_LINE, lw=LW, alpha=0.35):
    for y0 in ys0:
        for y1 in ys1:
            ax.plot([x0 + R, x1 - R], [y0, y1],
                    color=colour, linewidth=lw, alpha=alpha, zorder=1)

def col_label(cx, top_y, lines, fontsize=7.5):
    ax.text(cx, top_y + 0.42, '\n'.join(lines), ha='center', va='bottom',
            fontsize=fontsize, family='serif', fontweight='bold', linespacing=1.3)

def col_sublabel(cx, bot_y, text, fontsize=6.5):
    ax.text(cx, bot_y - 0.42, text, ha='center', va='top',
            fontsize=fontsize, family='serif', color='#666666', style='italic')

X_IN, X_H1, X_H2, X_Q, X_BOX = 1.1, 3.2, 5.3, 7.4, 9.6

# Input
N_IN_SHOWN = 5; y_in_top = 4.5; y_in_step = 0.70
ys_in = [y_in_top - i * y_in_step for i in range(N_IN_SHOWN)]
draw_nodes(X_IN, ys_in, C_INPUT)
dot_y = (ys_in[2] + ys_in[3]) / 2
for dy in [-0.18, 0.0, 0.18]:
    ax.plot(X_IN, dot_y + dy, 'o', color='#AAAAAA', markersize=1.8, zorder=4)
col_label(X_IN, y_in_top, ['State', 'Input']); col_sublabel(X_IN, ys_in[-1], '705-D')
for y, lbl in zip(ys_in, ['Feature 1','Feature 2','Feature 3','...','Feature 705']):
    ax.text(X_IN - R - 0.12, y, lbl, ha='right', va='center',
            fontsize=6.5, family='serif', color='#333333')

# Hidden 1
N_H = 6; y_h_top = 4.55; y_h_step = 0.62
ys_h1 = [y_h_top - i * y_h_step for i in range(N_H)]
connect(X_IN, ys_in, X_H1, ys_h1); draw_nodes(X_H1, ys_h1, C_HIDDEN)
col_label(X_H1, y_h_top, ['Hidden', 'Layer 1']); col_sublabel(X_H1, ys_h1[-1], '64 units, ReLU')

# Hidden 2
ys_h2 = [y_h_top - i * y_h_step for i in range(N_H)]
connect(X_H1, ys_h1, X_H2, ys_h2); draw_nodes(X_H2, ys_h2, C_HIDDEN)
col_label(X_H2, y_h_top, ['Hidden', 'Layer 2']); col_sublabel(X_H2, ys_h2[-1], '64 units, ReLU')

# Q-values
N_Q = 9; y_q_top = 4.55; y_q_step = (4.55 - 1.45) / (N_Q - 1)
ys_q = [y_q_top - i * y_q_step for i in range(N_Q)]
connect(X_H2, ys_h2, X_Q, ys_q); draw_nodes(X_Q, ys_q, C_OUTPUT)
col_label(X_Q, y_q_top, ['Q-Values']); col_sublabel(X_Q, ys_q[-1], '9 actions')
for y, lbl in zip(ys_q, [f'Q(a{i+1})' for i in range(9)]):
    ax.text(X_Q + R + 0.08, y, lbl, ha='left', va='center',
            fontsize=6.0, family='serif', color='#555555', style='italic')

# argmax arrow + selected action box
mid_q = (ys_q[0] + ys_q[-1]) / 2
ax.annotate('', xy=(X_BOX - 0.55, mid_q), xytext=(X_Q + R + 0.55, mid_q),
            arrowprops=dict(arrowstyle='->', color=C_ARROW, lw=1.0))
ax.text((X_Q + R + 0.55 + X_BOX - 0.55) / 2, mid_q + 0.22, 'argmax',
        ha='center', va='bottom', fontsize=6.5, family='serif',
        color=C_ARROW, style='italic')
box_cx, box_cy, box_w, box_h = X_BOX + 0.2, mid_q, 1.55, 0.70
ax.add_patch(mpatches.FancyBboxPatch(
    (box_cx - box_w/2, box_cy - box_h/2), box_w, box_h,
    boxstyle='round,pad=0.05',
    facecolor=C_SELECT, edgecolor='#B8860B', linewidth=1.0, zorder=3))
ax.text(box_cx, box_cy + 0.10, 'Selected Template', ha='center', va='center',
        fontsize=7.0, family='serif', fontweight='bold', color='#333333')
ax.text(box_cx, box_cy - 0.15, r'$a^* = \arg\max_a Q(s,a)$',
        ha='center', va='center', fontsize=6.5, family='serif', color='#555555')
ax.text(box_cx, box_cy + box_h/2 + 0.44, 'Action\nSelection',
        ha='center', va='bottom', fontsize=7.5,
        family='serif', fontweight='bold', linespacing=1.3)

ax.text(5.5, 0.62,
        u'ε-greedy exploration during training: with probability ε a random template is chosen; otherwise argmax is applied.',
        ha='center', va='center', fontsize=6.5, family='serif',
        color='#555555', style='italic')
ax.text(5.5, 5.28, 'Figure 6: DQN Q-network architecture',
        ha='center', va='bottom', fontsize=9, family='serif', style='italic')

plt.tight_layout(pad=0.3)
out = FIG_DIR / 'figure6_dqn_architecture.jpg'
plt.savefig(out, bbox_inches='tight', dpi=300, facecolor=BG)
plt.show(); print(f'Saved {out}')


## Figure 7 — Pipeline A fixed train/validation/test split


In [ ]:
COLOUR = {'train': '#4472C4', 'val': '#ED7D31', 'test': '#FFC000'}
TRAIN, VAL, TEST = (1999, 2018), (2019, 2021), (2022, 2024)

fig, ax = plt.subplots(figsize=(10, 2.4), dpi=300)
ax.set_xlim(1998.5, 2025.5); ax.set_ylim(0, 1.5); ax.axis('off')

def draw_block(ax, x0, x1, colour):
    width = x1 - x0 + 1
    ax.add_patch(plt.Rectangle((x0 - 0.5, 0.35), width, 0.5,
                                facecolor=colour, edgecolor='white', linewidth=0.8))

draw_block(ax, *TRAIN, COLOUR['train'])
draw_block(ax, *VAL,   COLOUR['val'])
draw_block(ax, *TEST,  COLOUR['test'])

ax.text((TRAIN[0] + TRAIN[1]) / 2, 0.60,
        f'Training\n{TRAIN[0]}–{TRAIN[1]}',
        ha='center', va='center', fontsize=8.5,
        family='serif', color='white', fontweight='bold', linespacing=1.4)

mid_val = (VAL[0] + VAL[1]) / 2
ax.annotate(f'Validation\n{VAL[0]}–{VAL[1]}',
            xy=(mid_val, 0.85), xytext=(mid_val - 1.2, 1.25),
            fontsize=7.5, family='serif', color=COLOUR['val'],
            fontweight='bold', ha='center', va='bottom', linespacing=1.3,
            arrowprops=dict(arrowstyle='-', color=COLOUR['val'],
                            lw=0.7, shrinkA=0, shrinkB=2))

mid_test = (TEST[0] + TEST[1]) / 2
ax.annotate(f'Test\n{TEST[0]}–{TEST[1]}',
            xy=(mid_test, 0.85), xytext=(mid_test + 1.2, 1.25),
            fontsize=7.5, family='serif', color='#B8860B',
            fontweight='bold', ha='center', va='bottom', linespacing=1.3,
            arrowprops=dict(arrowstyle='-', color='#B8860B',
                            lw=0.7, shrinkA=0, shrinkB=2))

for y in range(1999, 2025, 3):
    ax.text(y, 0.27, str(y), ha='center', va='top', fontsize=7, family='serif',
            color='#555555')
    ax.plot([y, y], [0.33, 0.35], color='#999999', linewidth=0.5)

ax.text((1998.5 + 2025.5) / 2, 1.45,
        'Figure 7: Pipeline A — fixed train–validation–test split',
        ha='center', va='top', fontsize=9, family='serif', fontstyle='italic')

plt.tight_layout()
out = FIG_DIR / 'figure7_pipeline_a.jpg'
plt.savefig(out, bbox_inches='tight', dpi=300, facecolor='white')
plt.show(); print(f'Saved {out}')


## Figure 8 — Pipeline B walk-forward windows


In [ ]:
COLOUR = {'train': '#4472C4', 'val': '#ED7D31', 'test': '#FFC000'}
EDGE   = '#FFFFFF'
YEARS = list(range(1999, 2024))

WINDOWS = {
    1: {'train': range(1999, 2014), 'val': range(2014, 2018), 'test': [2018]},
    2: {'train': range(2000, 2015), 'val': range(2015, 2019), 'test': [2019]},
    3: {'train': range(2001, 2016), 'val': range(2016, 2020), 'test': [2020]},
    4: {'train': range(2002, 2017), 'val': range(2017, 2021), 'test': [2021]},
    5: {'train': range(2003, 2018), 'val': range(2018, 2022), 'test': [2022]},
    6: {'train': range(2004, 2019), 'val': range(2019, 2023), 'test': [2023]},
}

def role(wd, year):
    if year in wd['train']: return 'train'
    if year in wd['val']:   return 'val'
    if year in wd['test']:  return 'test'
    return None

n_rows, n_cols = len(YEARS), 6
fig, ax = plt.subplots(figsize=(4.5, 8), dpi=300)
ax.set_xlim(0, n_cols); ax.set_ylim(0, n_rows)
ax.set_aspect('equal'); ax.axis('off')

for row, year in enumerate(YEARS):
    y = n_rows - row - 1
    for col, w in enumerate(range(1, 7)):
        r = role(WINDOWS[w], year)
        fc = COLOUR[r] if r else '#F2F2F2'
        ax.add_patch(plt.Rectangle((col, y), 1, 1,
                                    facecolor=fc, edgecolor=EDGE, linewidth=0.4))

for row, year in enumerate(YEARS):
    y = n_rows - row - 1 + 0.5
    ax.text(-0.15, y, str(year), ha='right', va='center',
            fontsize=7, family='serif')

for col, w in enumerate(range(1, 7)):
    ax.text(col + 0.5, n_rows + 0.25, str(w),
            ha='center', va='bottom', fontsize=8,
            family='serif', fontweight='bold')

ax.text(3.0, n_rows + 0.85, 'Window', ha='center', va='bottom',
        fontsize=8, family='serif', style='italic')

patches = [
    mpatches.Patch(facecolor=COLOUR['train'], edgecolor='grey', linewidth=0.4, label='Training set'),
    mpatches.Patch(facecolor=COLOUR['val'],   edgecolor='grey', linewidth=0.4, label='Validation set'),
    mpatches.Patch(facecolor=COLOUR['test'],  edgecolor='grey', linewidth=0.4, label='Test set'),
]
ax.legend(handles=patches, loc='upper left',
          bbox_to_anchor=(0.0, -0.01),
          fontsize=7, framealpha=0.8,
          handlelength=1.0, handleheight=0.9,
          borderpad=0.5, labelspacing=0.4)

ax.text(3.0, n_rows + 1.55,
        'Figure 8: Pipeline B — walk-forward windows',
        ha='center', va='bottom', fontsize=9, family='serif', fontstyle='italic')

plt.tight_layout()
out = FIG_DIR / 'figure8_pipeline_b.jpg'
plt.savefig(out, bbox_inches='tight', dpi=300, facecolor='white')
plt.show(); print(f'Saved {out}')


## Figure 9 — Pipeline A test-set equity curves (2022–2024)

Top panel: cumulative portfolio value ($1 invested) for all five strategies.
Bottom panel: drawdown of the two RL agents (mean across 3 random seeds, with
a ±1 σ shaded band).


In [ ]:
C_BH, C_EW, C_MC = '#808080', '#4472C4', '#70AD47'
C_PPO, C_DQN     = '#C00000', '#ED7D31'

df = pd.read_csv(PIPE_A_RET_CSV).dropna().reset_index(drop=True)
n = len(df)

raw_dates = pd.read_csv(PRICES_GZ, usecols=['date'])
raw_dates['date'] = pd.to_datetime(raw_dates['date'])
test_dates = (raw_dates['date']
              .drop_duplicates().sort_values()
              .loc[lambda s: (s >= '2022-01-01') & (s <= '2024-12-31')]
              .reset_index(drop=True))
dates = test_dates.iloc[:n].values

def cumret(s): return (1 + s).cumprod().values
def drawdown(arr):
    peak = np.maximum.accumulate(arr)
    return (arr - peak) / peak

cum_bh = cumret(df['Buy_Hold'])
cum_ew = cumret(df['Equal_Weight'])
cum_mc = cumret(df['MktCap_Weight'])

ppo_seeds = ['PPO_s42', 'PPO_s123', 'PPO_s456']
dqn_seeds = ['DQN_s42', 'DQN_s123', 'DQN_s456']

ppo_cum = np.stack([cumret(df[s]) for s in ppo_seeds])
dqn_cum = np.stack([cumret(df[s]) for s in dqn_seeds])

ppo_mean, ppo_std = ppo_cum.mean(0), ppo_cum.std(0)
dqn_mean, dqn_std = dqn_cum.mean(0), dqn_cum.std(0)

ppo_dd = np.stack([drawdown(cumret(df[s])) for s in ppo_seeds]).mean(0)
dqn_dd = np.stack([drawdown(cumret(df[s])) for s in dqn_seeds]).mean(0)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6.2),
                                gridspec_kw={'height_ratios': [3, 1]}, sharex=True)
fig.subplots_adjust(hspace=0.06)

ax1.plot(dates, cum_bh, color=C_BH, lw=0.9, ls='--', label='Buy-and-Hold',  zorder=3)
ax1.plot(dates, cum_ew, color=C_EW, lw=0.9, ls='--', label='Equal-Weight',  zorder=3)
ax1.plot(dates, cum_mc, color=C_MC, lw=0.9, ls='--', label='Mkt-Cap Weight',zorder=3)
ax1.fill_between(dates, ppo_mean - ppo_std, ppo_mean + ppo_std,
                 color=C_PPO, alpha=0.12, zorder=2)
ax1.plot(dates, ppo_mean, color=C_PPO, lw=1.5, label='PPO (mean ±1σ, 3 seeds)', zorder=4)
ax1.fill_between(dates, dqn_mean - dqn_std, dqn_mean + dqn_std,
                 color=C_DQN, alpha=0.12, zorder=2)
ax1.plot(dates, dqn_mean, color=C_DQN, lw=1.5, label='DQN (mean ±1σ, 3 seeds)', zorder=4)
ax1.axhline(1.0, color='#555555', lw=0.55, ls=':', zorder=1)
ax1.set_ylabel('Cumulative portfolio value ($1 invested)',
               fontsize=8.5, family='serif', labelpad=6)
ax1.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f'${y:.2f}'))
ax1.legend(loc='upper left', fontsize=7.5, framealpha=0.92,
           edgecolor='#cccccc', prop={'family': 'serif', 'size': 7.5})
ax1.grid(axis='y', ls=':', lw=0.5, color='#cccccc')
ax1.tick_params(labelsize=8)
for sp in ['top', 'right']:
    ax1.spines[sp].set_visible(False)

ax2.fill_between(dates, ppo_dd, 0, color=C_PPO, alpha=0.30, label='PPO drawdown')
ax2.fill_between(dates, dqn_dd, 0, color=C_DQN, alpha=0.30, label='DQN drawdown')
ax2.plot(dates, ppo_dd, color=C_PPO, lw=0.85)
ax2.plot(dates, dqn_dd, color=C_DQN, lw=0.85)
ax2.axhline(0, color='#555555', lw=0.55, ls=':')
dd_floor = min(ppo_dd.min(), dqn_dd.min())
ax2.set_ylim(dd_floor * 1.08, 0.01)
ax2.set_ylabel('Drawdown (RL Agents)', fontsize=8.5, family='serif', labelpad=6)
ax2.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f'{y:.0%}'))
ax2.legend(loc='lower left', fontsize=7.5, framealpha=0.92,
           edgecolor='#cccccc', prop={'family': 'serif', 'size': 7.5})
ax2.grid(axis='y', ls=':', lw=0.5, color='#cccccc')
ax2.tick_params(labelsize=8)
for sp in ['top', 'right']:
    ax2.spines[sp].set_visible(False)
ax2.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 4, 7, 10]))
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=0, ha='center', fontsize=8)

plt.tight_layout()
fig.text(0.5, -0.02,
         'Figure 9: Cumulative portfolio value for all five strategies over the Pipeline A test period '
         '(2022–2024). PPO and DQN show the cross-seed mean; shaded bands mark ±1 standard deviation.',
         ha='center', va='top', fontsize=9, family='serif', fontstyle='italic', wrap=True)

out = FIG_DIR / 'figure9_test_a_equity.jpg'
plt.savefig(out, bbox_inches='tight', dpi=300, facecolor='white')
plt.show(); print(f'Saved {out}')


## Figure 10 — Pipeline B per-window Sharpe ratios (2018–2023)


In [ ]:
C_BH, C_EW, C_MC = '#808080', '#4472C4', '#70AD47'
C_PPO, C_DQN     = '#C00000', '#ED7D31'

df = pd.read_csv(PIPE_B_RES_CSV)
YEAR_MAP = {1: 2018, 2: 2019, 3: 2020, 4: 2021, 5: 2022, 6: 2023}
df['year'] = df['window'].map(YEAR_MAP)

strategies = [
    ('Buy_Hold',     C_BH,  'Buy-and-Hold'),
    ('Equal_Weight', C_EW,  'Equal-Weight'),
    ('MktCap_Weight',C_MC,  'Mkt-Cap Weight'),
    ('PPO',          C_PPO, 'PPO'),
    ('DQN',          C_DQN, 'DQN'),
]
years = [2018, 2019, 2020, 2021, 2022, 2023]
n_strategies, n_years = len(strategies), len(years)

group_width = 0.75
bar_width   = group_width / n_strategies
x = np.arange(n_years)
offsets = np.linspace(-(group_width - bar_width) / 2,
                       (group_width - bar_width) / 2,
                       n_strategies)

fig, ax = plt.subplots(figsize=(10, 4.6), dpi=300)
for i, (col, color, label) in enumerate(strategies):
    heights = [
        df.loc[(df['algorithm'] == col) & (df['year'] == yr), 'sharpe'].values[0]
        for yr in years
    ]
    ax.bar(x + offsets[i], heights, width=bar_width,
           color=color, alpha=0.88, zorder=3, label=label)

ax.axhline(0, color='#333333', lw=0.8, zorder=4)
for xi in x[:-1]:
    ax.axvline(xi + 0.5, color='#dddddd', lw=0.7, ls='--', zorder=2)

ax.set_xticks(x)
ax.set_xticklabels([str(y) for y in years], fontsize=9, family='serif')
ax.set_ylabel('Sharpe Ratio (annualised)', fontsize=8.5, family='serif', labelpad=6)
ax.tick_params(axis='y', labelsize=8)
ax.grid(axis='y', ls=':', lw=0.5, color='#cccccc', zorder=1)
for sp in ['top', 'right']:
    ax.spines[sp].set_visible(False)

legend_handles = [
    mpatches.Patch(color=color, alpha=0.88, label=label)
    for _, color, label in strategies
]
ax.legend(handles=legend_handles, loc='upper left', fontsize=7.5,
          framealpha=0.92, edgecolor='#cccccc',
          prop={'family': 'serif', 'size': 7.5})

plt.tight_layout()
fig.text(0.5, -0.02,
         'Figure 10: Pipeline B test-year Sharpe ratios across six walk-forward windows (2018–2023). '
         'Each bar group corresponds to one test year; all five strategies are shown.',
         ha='center', va='top', fontsize=9, family='serif', fontstyle='italic', wrap=True)

out = FIG_DIR / 'figure10_test_b_sharpe_bar.jpg'
plt.savefig(out, bbox_inches='tight', dpi=300, facecolor='white')
plt.show(); print(f'Saved {out}')


## Equation renders

Static mathtext renders of the four equations cited in Chapter 4. Numbered to
match the body of the thesis (1: z-score, 2: clipped reward, 3: annualised
Sharpe, 4: paired-test hypotheses).


In [ ]:
# Equation (1) — z-score standardisation
fig, ax = plt.subplots(figsize=(9, 1.4), dpi=300)
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
ax.text(0.5, 0.5,
        r"$\tilde{x}_{i,t} \;=\; \dfrac{x_{i,t} \,-\, \mu_{i}^{\,\mathrm{train}}}{\sigma_{i}^{\,\mathrm{train}}}$",
        ha="center", va="center", fontsize=22, family="serif")
ax.text(0.98, 0.5, "(1)", ha="right", va="center", fontsize=14, family="serif")
out = FIG_DIR / "eq_zscore.jpg"
plt.savefig(out, bbox_inches="tight", dpi=300, facecolor="white", pad_inches=0.2)
plt.show(); print(f"Saved {out}")


In [ ]:
# Equation (2) — clipped reward signal
fig, ax = plt.subplots(figsize=(9, 2.2), dpi=300)
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
ax.text(0.5, 0.68,
        r"$r_t \;=\; \mathrm{clip}\!\left(100\cdot r^{\,p}_{t+1},\;-10,\;10\right)$",
        ha="center", va="center", fontsize=22, family="serif")
ax.text(0.5, 0.28,
        r"$r^{\,p}_{t+1} \;=\; \mathbf{w}_t^{\top}\mathbf{R}_{t+1} \;-\; c\,\|\Delta\mathbf{w}\|_1$",
        ha="center", va="center", fontsize=18, family="serif", color="#444444")
ax.text(0.98, 0.5, "(2)", ha="right", va="center", fontsize=14, family="serif")
out = FIG_DIR / "eq_reward.jpg"
plt.savefig(out, bbox_inches="tight", dpi=300, facecolor="white", pad_inches=0.2)
plt.show(); print(f"Saved {out}")


In [ ]:
# Equation (3) — annualised Sharpe ratio
fig, ax = plt.subplots(figsize=(9, 1.4), dpi=300)
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
ax.text(0.5, 0.5,
        r"$\mathrm{Sharpe}_{\,\mathrm{annualised}} \;=\; \dfrac{\bar{r}}{\sigma_{r}}\,\cdot\,\sqrt{252}$",
        ha="center", va="center", fontsize=22, family="serif")
ax.text(0.98, 0.5, "(3)", ha="right", va="center", fontsize=14, family="serif")
out = FIG_DIR / "eq_sharpe.jpg"
plt.savefig(out, bbox_inches="tight", dpi=300, facecolor="white", pad_inches=0.2)
plt.show(); print(f"Saved {out}")


In [ ]:
# Equation (4) — paired-test hypotheses
fig, ax = plt.subplots(figsize=(9, 2.0), dpi=300)
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
ax.text(0.5, 0.68,
        r"$H_0\colon\; \mathrm{Sharpe}_{\mathrm{agent}} \;\leq\; \mathrm{Sharpe}_{\mathrm{EW}}$",
        ha="center", va="center", fontsize=20, family="serif")
ax.text(0.5, 0.28,
        r"$H_1\colon\; \mathrm{Sharpe}_{\mathrm{agent}} \;>\; \mathrm{Sharpe}_{\mathrm{EW}}$",
        ha="center", va="center", fontsize=20, family="serif")
ax.text(0.98, 0.5, "(4)", ha="right", va="center", fontsize=14, family="serif")
out = FIG_DIR / "eq_hypotheses.jpg"
plt.savefig(out, bbox_inches="tight", dpi=300, facecolor="white", pad_inches=0.2)
plt.show(); print(f"Saved {out}")
